# Lab - BUILD: An Agentic System That Survives Repeats

Three tasks that turn the loop into one system with case facts, a hook and a handoff - and make it hold up in production, where workers restart and customers write the same thing twice.

| Task | What you learn |
| --- | --- |
| 1 | Case facts filled by the tools and saved to a file on every change |
| 2 | The hook's three rules: verified, looked up, and one ticket per case |
| 3 | The customer writes again - the hook points to the ticket, no second one |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Use the tools for customers, orders and returns; the tools "
    "check the policy and tell you what is missing. If a tool "
    "refuses and the customer still needs help, hand the case "
    "to a person with escalate. Answer in two sentences."
)

CUSTOMERS = {"alex@example.com": "CUS-1001"}
ORDERS = {"ORD-87654321": {"item": "lamp", "status": "delivered",
                           "delivered_days_ago": 45, "customer_id": "CUS-1001"}}


def tool(name, description, field):
    return {"name": name, "description": description,
            "input_schema": {"type": "object", "properties": {field: {"type": "string"}},
                             "required": [field]}}


TOOLS = [
    tool("get_customer", "Verify a customer by email. Returns the customer ID.", "email"),
    tool("lookup_order", "Look up an order by its ID.", "order_id"),
    tool("start_return", "Start a return for a delivered order inside the 30-day window.",
         "order_id"),
    tool("escalate", "Hand the case to a human agent with a short reason. Use it when a "
         "tool refused and the customer still needs help.", "reason"),
]


def start_return(order_id):   # as in the tool errors lab
    order = ORDERS[order_id]
    if order["delivered_days_ago"] > 30:
        return {"isError": True, "errorCategory": "business", "isRetryable": False,
                "customerMessage": "Past the 30-day return window."}
    return {"return_label": "RET-2048"}


CASE_FILE = Path("case.json")


def save_case():   # the case lives in a file, not only in memory
    CASE_FILE.write_text(json.dumps(CASE))


def load_case():
    return json.loads(CASE_FILE.read_text())

---

## Task 1 - Case facts that survive a restart

In [ ]:
# ============================================================
# TASK 1 - Case facts that survive a restart
# ============================================================
#
# WHAT TO DO
#   Keep the case in one dictionary - customer, order, ticket -
#   and let each tool record what it found and save the case.
#   escalate opens a ticket and keeps it on the case.
#
# WHY IT MATTERS
#   What the conversation established must not depend on what
#   Claude remembers or repeats. The tools fill the case as they
#   run - and a case that lives only in memory is lost when the
#   worker restarts, or invisible to the next process that
#   picks the customer up. Saved on every change, the facts are
#   there for whoever reads them next - including the hook.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Second, the case: a small dictionary".
#
# HOW TO DO IT
#   Three blanks:
#
#       customer_id   what get_customer found
#       order_id      the order that was looked up
#       ticket        the ticket escalate opened
# ============================================================

CASE = {"customer_id": None, "order_id": None, "ticket": None}
TICKETS = []   # every ticket ever opened

try:
    # BEGIN SOLUTION
    def get_customer(email):
        customer_id = CUSTOMERS.get(email)
        if customer_id is None:
            return {"found": False}
        CASE["customer_id"] = customer_id
        save_case()
        return {"found": True, "customer_id": customer_id}

    def lookup_order(order_id):
        order = ORDERS.get(order_id)
        if order is None:
            return {"found": False, "message": "No order with this ID."}
        CASE["order_id"] = order_id
        save_case()
        return {"found": True, "order": order}

    def escalate(reason):
        ticket = f"ESC-{507 + len(TICKETS)}"
        TICKETS.append({"ticket": ticket, "reason": reason})
        CASE["ticket"] = ticket
        save_case()
        return {"ticket": ticket, "reason": reason}
    # SCAFFOLD: def get_customer(email):
    # SCAFFOLD:     customer_id = CUSTOMERS.get(email)
    # SCAFFOLD:     if customer_id is None:
    # SCAFFOLD:         return {"found": False}
    # SCAFFOLD:     CASE["customer_id"] = ...   # customer_id
    # SCAFFOLD:     save_case()
    # SCAFFOLD:     return {"found": True, "customer_id": customer_id}
    # SCAFFOLD:
    # SCAFFOLD: def lookup_order(order_id):
    # SCAFFOLD:     order = ORDERS.get(order_id)
    # SCAFFOLD:     if order is None:
    # SCAFFOLD:         return {"found": False, "message": "No order with this ID."}
    # SCAFFOLD:     CASE["order_id"] = ...   # order_id
    # SCAFFOLD:     save_case()
    # SCAFFOLD:     return {"found": True, "order": order}
    # SCAFFOLD:
    # SCAFFOLD: def escalate(reason):
    # SCAFFOLD:     ticket = f"ESC-{507 + len(TICKETS)}"
    # SCAFFOLD:     TICKETS.append({"ticket": ticket, "reason": reason})
    # SCAFFOLD:     CASE["ticket"] = ...   # ticket
    # SCAFFOLD:     save_case()
    # SCAFFOLD:     return {"ticket": ticket, "reason": reason}
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
TOOL_FUNCTIONS = {"get_customer": get_customer, "lookup_order": lookup_order,
                  "start_return": start_return, "escalate": escalate}

check("case_record", case=CASE, get_customer=get_customer, lookup_order=lookup_order,
      escalate=escalate, load_case=load_case)

---

## Task 2 - Three rules before any tool runs

In [ ]:
# ============================================================
# TASK 2 - Three rules before any tool runs
# ============================================================
#
# WHAT TO DO
#   Finish before_tool(), the hook. It keeps the lecture's two
#   rules - verified first, look the order up before a return -
#   and adds a third: one ticket per case. A second escalate
#   for a case that already has a ticket is refused, and the
#   refusal names the ticket.
#
# WHY IT MATTERS
#   Agents repeat themselves: the customer writes again, a
#   request is retried after a timeout, the model decides to
#   escalate once more. Without a rule, every repeat is a new
#   action - a second ticket, a second refund. The hook reads
#   the saved case before every call, so a repeated action is
#   refused, not performed twice.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Third, the hook".
#
# HOW TO DO IT
#   Three blanks:
#
#       "escalate"   the action that must happen once per case
#       "ticket"     the case already has one
#       False        retrying cannot change it
# ============================================================

# BEGIN SOLUTION
def before_tool(name, args):
    """The hook: runs before every tool call and can block it."""
    if name != "get_customer" and CASE["customer_id"] is None:
        return {"isError": True, "errorCategory": "permission",
                "customerMessage": "Please give your email first."}
    if name == "start_return" and args["order_id"] != CASE["order_id"]:
        return {"isError": True, "errorCategory": "validation",
                "customerMessage": "Look up the order first."}
    if name == "escalate" and CASE["ticket"]:
        return {"isError": True, "errorCategory": "business", "isRetryable": False,
                "customerMessage": f"This case is already with a colleague: ticket {CASE['ticket']}."}
    return None
# SCAFFOLD: def before_tool(name, args):
# SCAFFOLD:     """The hook: runs before every tool call and can block it."""
# SCAFFOLD:     if name != "get_customer" and CASE["customer_id"] is None:
# SCAFFOLD:         return {"isError": True, "errorCategory": "permission",
# SCAFFOLD:                 "customerMessage": "Please give your email first."}
# SCAFFOLD:     if name == "start_return" and args["order_id"] != CASE["order_id"]:
# SCAFFOLD:         return {"isError": True, "errorCategory": "validation",
# SCAFFOLD:                 "customerMessage": "Look up the order first."}
# SCAFFOLD:     if name == ... and CASE[...]:   # "escalate", "ticket"
# SCAFFOLD:         return {"isError": True, "errorCategory": "business", "isRetryable": ...,   # False
# SCAFFOLD:                 "customerMessage": f"This case is already with a colleague: ticket {CASE['ticket']}."}
# SCAFFOLD:     return None
# END SOLUTION: replace each ... below with the value named beside it

try:
    print(before_tool("lookup_order", {"order_id": "ORD-87654321"}))
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("hook_rules", before_tool=before_tool, case=CASE)

---

## Task 3 - The customer writes again: no second ticket

In [ ]:
# ============================================================
# TASK 3 - The customer writes again: no second ticket
# ============================================================
#
# WHAT TO DO
#   Run the loop through the hook for three messages: the lamp
#   without an email, the same customer with their email, and
#   the customer writing again about the same lamp. Count the
#   tickets, and read the case back from its file.
#
# WHY IT MATTERS
#   The first message is blocked - nothing looked up, nothing
#   started. The second is verified, refused by the policy and
#   handed to a person. The third is the real world: the
#   customer asks again, and the agent, reasoning the same way,
#   tries to escalate again. The hook's third rule turns that
#   into a pointer to the existing ticket.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Here is the second case, step by step".
#
# HOW TO DO IT
#   Three blanks:
#
#       text          each customer message, run through the loop
#       TICKETS       how many tickets were opened
#       load_case()   the case, read back from case.json
# ============================================================


def run(text):   # provided: the loop - every tool call goes through the hook first
    messages = [{"role": "user", "content": text}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=600, output_config={"effort": "low"},
            system=SYSTEM, tools=TOOLS, messages=messages)
        if response.stop_reason != "tool_use":
            return response, messages
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = before_tool(block.name, block.input) or \
                    TOOL_FUNCTIONS[block.name](**block.input)
                print(f"  {block.name}({block.input}) -> {output}")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(output),
                                "is_error": bool(output.get("isError"))})
        messages.append({"role": "user", "content": results})


MESSAGES = [
    "I want to return the lamp from order ORD-87654321.",
    "Hi, I am alex@example.com. The lamp from order ORD-87654321 arrived broken "
    "and I want to return it.",
    "Hi, I am alex@example.com again. Any news? I still want to return the lamp "
    "from order ORD-87654321.",
]

CASE.update(customer_id=None, order_id=None, ticket=None); TICKETS.clear(); save_case()
try:
    # BEGIN SOLUTION
    runs = []
    for text in MESSAGES:
        print("Customer:", text)
        runs.append(run(text))
        print("ShopAssist:", runs[-1][0].content[0].text, "\n")
    print("tickets opened:", len(TICKETS))
    print("case on disk:", load_case())
    # SCAFFOLD: runs = []
    # SCAFFOLD: for text in MESSAGES:
    # SCAFFOLD:     print("Customer:", text)
    # SCAFFOLD:     runs.append(run(...))   # text
    # SCAFFOLD:     print("ShopAssist:", runs[-1][0].content[0].text, "\n")
    # SCAFFOLD: print("tickets opened:", len(...))   # TICKETS
    # SCAFFOLD: print("case on disk:", ...)   # load_case()
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("no_duplicates", runs=runs, tickets=TICKETS)

---

## Done

Keep the case facts in your code and your storage, let a hook check every tool call before it runs - including whether this exact action already happened - and give the loop a handoff tool for the cases the policy cannot settle.